In [0]:
from pyspark.sql import functions as F


def create_file_bronze_table(
    source_folder: str,
    table_name: str,
    source_system: str,
) -> None:
    files = (
        spark.read
        .format("binaryFile")
        .option("pathGlobFilter", "*.json")
        .load(source_folder)
    )

    bronze = files.select(
        F.element_at(F.split("path", "/"), -1).alias("source_file"),
        F.col("path").alias("source_path"),
        F.lit(source_system).alias("source_system"),
        F.col("modificationTime").alias("source_modified_at"),
        F.col("length").alias("source_size_bytes"),
        F.current_timestamp().alias("ingested_at"),
        F.sha2("content", 256).alias("record_hash"),
        F.col("content").cast("string").alias("raw_payload"),
    )

    (
        bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    row_count = spark.table(table_name).count()
    print(f"{table_name}: {row_count} filer")

In [0]:
create_file_bronze_table(
    source_folder="/Volumes/clubdata/bronze/landing/geocoding",
    table_name="clubdata.bronze.geocoding_raw",
    source_system="Nominatim",
)

create_file_bronze_table(
    source_folder="/Volumes/clubdata/bronze/landing/weather/sources",
    table_name="clubdata.bronze.weather_sources_raw",
    source_system="Frost",
)

create_file_bronze_table(
    source_folder="/Volumes/clubdata/bronze/landing/weather/observations",
    table_name="clubdata.bronze.weather_observations_raw",
    source_system="Frost",
)

In [0]:
%sql

SELECT
    'geocoding_raw' AS table_name,
    COUNT(*) AS rows,
    COUNT(DISTINCT record_hash) AS distinct_files
FROM clubdata.bronze.geocoding_raw

UNION ALL

SELECT
    'weather_sources_raw',
    COUNT(*),
    COUNT(DISTINCT record_hash)
FROM clubdata.bronze.weather_sources_raw

UNION ALL

SELECT
    'weather_observations_raw',
    COUNT(*),
    COUNT(DISTINCT record_hash)
FROM clubdata.bronze.weather_observations_raw;